# Suite XPRT — Printing C++ translation units from the store

`Programs.Cpp.Print` prints a translation unit as the store holds it: a reference object that
`Programs.Framework.TheWorkspace` owns by its path, holding its tree as one value, its `body`, each declaration and
statement a branch of the union `Programs.Cpp.Statement`. Its printer is C's (`Programs.C.Print.Printer`) over C++'s
terms, which mbse-expressions' C family printer writes, with C++'s statements, declarations and members: where a
lambda, a statement expression, a class or a conditional directive holds statements, members or a group (its `block`,
`members` or `group`), they print within the term, at the level of the statement that holds it. The layout is C's,
and 0.3's for what C++ adds: a namespace's, a linkage specification's and an `export`'s braced declarations one level
in, a template's head on a line of its own, access specifiers one level out, `} catch (...) {`, constructors' member
initializers after their declarator.

Each case builds its translation unit in a transaction of its own, its tree written in the builders' own syntax (`a`
below writes the name `a` as an expression, `int` the specifier `int`), and checks the printed text. The cases port
0.3's C++ cases of PRT (`04_Print.ipynb`); its cases of expressions alone are mbse-expressions' own suite's. The case
setup is below.

In [ ]:
from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import Cpp
from mbse.Programs.Cpp import Parse, Print, Statements
from mbse.Programs.Framework import Model
from support import raises

store = Cpp.store()
R = store.registered


def build(path, body):
    """The translation unit at `path`, built in a transaction of its own: `body` its declarations' writers."""
    store.transaction(lambda t: t.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"ws": R("Programs.Framework.Workspace"), "unit": R("Programs.Cpp.TranslationUnit")})
        .spec([lambda _: _.ws.files({path: lambda e: e.file(_.unit)}), lambda _: _.unit.body(body)])]))]))


def printed(path, body):
    """The text of the translation unit built at `path`."""
    build(path, body)
    return Print.print_module(store, path)


def within(path, statements):
    """The text of the function `void f()` whose body holds `statements`, built at `path`."""
    return printed(path, [function("f", statements)])


def reprinted(text):
    """The text `text` parses to, printed."""
    store.transaction(Parse.parse(text, "reprinted.cpp"))
    return Print.print_module(store, "reprinted.cpp")


# The writers the cases share, in the builders' syntax: names, terms, specifiers, declarators and statements
def name(spelling):
    return lambda i: i.spelling(spelling)


def ref(spelling):
    """The name `spelling` used as an expression."""
    return lambda t: t.idexpression(lambda e: e.name(lambda n: n.identifier(name(spelling))))


def number(value):
    return lambda t: t.literal(lambda l: l.value(value))


def keyword(word):
    return lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))


def declared(spelling):
    return lambda d: d.iddeclarator(lambda i: i.name(lambda n: n.identifier(name(spelling))))


def function_of(spelling):
    """The declarator `spelling()`."""
    return lambda d: d.functiondeclarator(lambda g: g.declarator(declared(spelling)))


def call(function_, *arguments):
    return lambda t: t.callexpression(lambda c: c.function(ref(function_)).arguments(list(arguments)))


def say(function_):
    """The statement `function_();`."""
    return lambda u: u.expressionstatement(lambda s: s.expression(call(function_)))


def block(statements):
    return lambda k: k.body(statements)


def function(spelling, statements, returns="void"):
    """`returns spelling() { statements }`."""
    return lambda u: u.functiondefinition(lambda f: f.specifiers([keyword(returns)]).declarator(
        function_of(spelling)).body(lambda b: b.compoundstatement(block(statements))))


def variable(spelling, initializer=None):
    """`int spelling = initializer;`, a declaration."""
    declarator = (lambda d: d.declarator(declared(spelling))) if initializer is None else (
        lambda d: d.declarator(declared(spelling)).initializer(initializer))
    return lambda u: u.simpledeclaration(lambda s: s.specifiers([integer]).declarators([declarator]))


a, b, c = ref("a"), ref("b"), ref("c")
integer = keyword("int")
compound = lambda statements: lambda k: k.compoundstatement(block(statements))  # noqa: E731
equal = lambda value: lambda i: i.equalinitializer(lambda e: e.value(value))  # noqa: E731

<a id="XPRT-01"></a>
## XPRT-01 · Every simple declaration and statement C++ adds prints alone, on its line

`;` alone, `co_return` with a value and without, a namespace's alias, a using-directive, a using-declaration of
several declarators, `typename` and a pack among them, `using enum`, an alias declaration, a concept, C++20's module
and import declarations and fragments, and explicit instantiations, `extern` and not; and C's kinds, by name, over
C++'s terms: a declaration whose declarators are initialized `= x`, `(x)` and `{x}`, with a trailing `requires`.

In [ ]:
qualified = lambda *names: lambda n: n.qualifiedname(lambda q: q.qualifiers([  # noqa: E731
    lambda x, s=s: x.identifier(name(s)) for s in names[:-1]]).name(lambda x: x.identifier(name(names[-1]))))
text = printed("simple.cpp", [
    lambda u: u.emptydeclaration(lambda s: s),
    lambda u: u.namespacealiasdefinition(lambda s: s.name(name("fs")).target(qualified("std", "filesystem"))),
    lambda u: u.usingdirective(lambda s: s.name(lambda n: n.identifier(name("std")))),
    lambda u: u.usingdeclaration(lambda s: s.declarators([
        lambda d: d.name(qualified("std", "vector")), lambda d: d.typename_keyword(True).name(qualified("T", "type")),
        lambda d: d.name(qualified("B", "f")).pack(True)])),
    lambda u: u.usingenumdeclaration(lambda s: s.type(lambda n: n.identifier(name("Color")))),
    lambda u: u.aliasdeclaration(lambda s: s.name(name("V")).type(lambda t: t.typeid(lambda i: i.specifiers([
        lambda x: x.namedtypespecifier(lambda n: n.name(qualified("std", "size_t")))])))),
    lambda u: u.conceptdefinition(lambda s: s.name(name("Small")).constraint(lambda t: t.binaryexpression(
        lambda x: x.left(a).operator("<").right(number(4))))),
    lambda u: u.globalmodulefragment(lambda s: s),
    lambda u: u.moduledeclaration(lambda s: s.export(True).name("shapes.core").partition("detail")),
    lambda u: u.importdeclaration(lambda s: s.name("std")),
    lambda u: u.importdeclaration(lambda s: s.export(True).partition("part")),
    lambda u: u.importdeclaration(lambda s: s.header("vector").system(True)),
    lambda u: u.importdeclaration(lambda s: s.header("legacy.h")),
    lambda u: u.privatemodulefragment(lambda s: s),
    lambda u: u.explicitinstantiation(lambda s: s.declaration(lambda d: d.specifiers([
        lambda x: x.classspecifier(lambda k: k.key("class").name(lambda n: n.templateid(lambda i: i.name(
            lambda x: x.identifier(name("V"))).arguments([lambda t: t.typeid(lambda i: i.specifiers([integer]))]))))]))),
    lambda u: u.explicitinstantiation(lambda s: s.extern(True).declaration(lambda d: d.specifiers([
        lambda x: x.classspecifier(lambda k: k.key("struct").name(lambda n: n.identifier(name("W"))))]))),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([integer]).declarators([
        lambda d: d.declarator(declared("x")).initializer(equal(number(1))),
        lambda d: d.declarator(declared("y")).initializer(lambda i: i.parenthesizedinitializer(
            lambda p: p.arguments([number(2)]))),
        lambda d: d.declarator(declared("z")).initializer(lambda i: i.initializerlist(lambda l: l.items([number(3)]))),
        lambda d: d.declarator(function_of("g")).requires(a)])),
])
assert text == """\
;
namespace fs = std::filesystem;
using namespace std;
using std::vector, typename T::type, B::f...;
using enum Color;
using V = std::size_t;
concept Small = a < 4;
module;
export module shapes.core:detail;
import std;
export import :part;
import <vector>;
import "legacy.h";
module :private;
template class V<int>;
extern template struct W;
int x = 1, y(2), z{3}, g() requires a;
"""
assert within("coreturn.cpp", [lambda u: u.coreturnstatement(lambda s: s.value(a)),
                                lambda u: u.coreturnstatement(lambda s: s)]) == (
    "void f() {\n    co_return a;\n    co_return;\n}\n")
print("ok")

<a id="XPRT-02"></a>
## XPRT-02 · Every compound statement C++ changes or adds prints, with its substatements

`if` with an initializer and a condition that declares, `if constexpr`, `if consteval` and `if !consteval`;
`switch` with an initializer; `while` whose condition declares; `for` whose initializer is an alias declaration;
range-based `for`, with an initializer and without; `try` with its handlers, and a handler alone.

In [ ]:
declaring = lambda u: u.simpledeclaration(lambda s: s.specifiers([integer]).declarators([  # noqa: E731
    lambda d: d.declarator(declared("x")).initializer(equal(call("g")))]))
handler = lambda parameter, statements: lambda h: h.parameter(parameter).body(block(statements))  # noqa: E731
ellipsis = lambda p: p.ellipsisparameter(lambda e: e)  # noqa: E731
caught = lambda p: p.parameterdeclaration(lambda q: q.specifiers([  # noqa: E731
    lambda s: s.cvqualifier(lambda x: x.keyword("const")), lambda s: s.namedtypespecifier(
        lambda n: n.name(lambda x: x.identifier(name("E"))))]).declarator(lambda d: d.referencedeclarator(
            lambda r: r.declarator(declared("e")))))
text = within("compound.cpp", [
    lambda u: u.ifstatement(lambda s: s.initializer(lambda i: i.simpledeclaration(lambda d: d.specifiers([integer])
        .declarators([lambda x: x.declarator(declared("v")).initializer(equal(number(1)))]))).condition(
        lambda c: c.expression(ref("v"))).consequence(compound([say("b")]))),
    lambda u: u.ifstatement(lambda s: s.condition(lambda c: c.simpledeclaration(lambda d: d.specifiers([integer])
        .declarators([lambda x: x.declarator(declared("x")).initializer(equal(call("g")))]))).consequence(say("b"))),
    lambda u: u.ifstatement(lambda s: s.constexpr(True).condition(lambda c: c.expression(a)).consequence(compound([]))
                            .alternative(compound([say("c")]))),
    lambda u: u.ifstatement(lambda s: s.consteval(True).consequence(compound([say("b")])).alternative(
        lambda e: e.ifstatement(lambda s: s.consteval(True).negated(True).consequence(compound([]))))),
    lambda u: u.switchstatement(lambda s: s.initializer(lambda i: i.expression(call("g"))).condition(
        lambda c: c.expression(a)).body(compound([lambda u: u.defaultstatement(lambda d: d.statement(
            lambda s: s.breakstatement(lambda x: x)))]))),
    lambda u: u.whilestatement(lambda s: s.condition(lambda c: c.simpledeclaration(lambda d: d.specifiers([integer])
        .declarators([lambda x: x.declarator(declared("y")).initializer(equal(call("g")))]))).body(say("b"))),
    lambda u: u.forstatement(lambda s: s.initializer(lambda i: i.aliasdeclaration(lambda x: x.name(name("T")).type(
        lambda t: t.typeid(lambda i: i.specifiers([integer]))))).body(compound([]))),
    lambda u: u.rangeforstatement(lambda s: s.declaration(lambda d: d.specifiers([
        lambda x: x.placeholdertypespecifier(lambda p: p)]).declarators([lambda x: x.declarator(
            lambda d: d.referencedeclarator(lambda r: r.declarator(declared("x"))))])).range(ref("xs")).body(
        compound([]))),
    lambda u: u.rangeforstatement(lambda s: s.initializer(lambda i: i.simpledeclaration(lambda d: d.specifiers([
        integer]).declarators([lambda x: x.declarator(declared("n")).initializer(equal(number(0)))]))).declaration(
        lambda d: d.specifiers([integer]).declarators([lambda x: x.declarator(declared("x"))])).range(ref("xs")).body(
        say("b"))),
    lambda u: u.trystatement(lambda s: s.body(block([say("b")])).handlers([handler(caught, [
        lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.throwexpression(lambda x: x)))]),
        handler(ellipsis, [])])),
])
assert text == """\
void f() {
    if (int v = 1; v) {
        b();
    }
    if (int x = g())
        b();
    if constexpr (a) {} else {
        c();
    }
    if consteval {
        b();
    } else if !consteval {}
    switch (g(); a) {
        default:
            break;
    }
    while (int y = g())
        b();
    for (using T = int;;) {}
    for (auto &x : xs) {}
    for (int n = 0; int x : xs)
        b();
    try {
        b();
    } catch (const E &e) {
        throw;
    } catch (...) {}
}
"""
build("handler.cpp", [lambda u: u.trystatement(lambda s: s.body(block([])).handlers([handler(ellipsis, [say("b")])]))])
with store.transaction():
    (tried,) = Objects.properties(store.singleton(Model.WORKSPACE).files({"path": "handler.cpp"}))["body"].values
    (alone,) = Objects.properties(Terms.kind_of(tried)[1])["handlers"].values
    assert Print.print_statement(alone) == "catch (...) {\n    b();\n}"  # a clause alone
print("ok")

<a id="XPRT-03"></a>
## XPRT-03 · Declarations that hold declarations: namespaces, nested and inline, linkage specifications and `export`, braced and not, templates and their `requires`, explicit specializations; functions' virt-specifiers, `requires`, member initializers, function-try-blocks, `= default` and `= delete`

In [ ]:
template = lambda parameters, declaration: lambda u: u.templatedeclaration(  # noqa: E731
    lambda s: s.parameters(parameters).declaration(declaration))
typename = lambda spelling: lambda p: p.typeparameter(lambda t: t.key("typename").name(name(spelling)))  # noqa: E731
text = printed("holding.cpp", [
    lambda u: u.namespacedefinition(lambda s: s.inline(True).names([lambda n: n.name(name("a")),
                                                                   lambda n: n.inline(True).name(name("b"))]).body([
        variable("x")])),
    lambda u: u.namespacedefinition(lambda s: s),
    lambda u: u.linkagespecification(lambda s: s.language("C").braced(True).body([variable("y")])),
    lambda u: u.linkagespecification(lambda s: s.language("C").body([variable("z")])),
    lambda u: u.exportdeclaration(lambda s: s.braced(True).body([variable("e")])),
    lambda u: u.exportdeclaration(lambda s: s.body([function("g", [])])),
    template([typename("T")], lambda d: d.functiondefinition(lambda f: f.specifiers([keyword("void")]).declarator(
        function_of("h")).requires(ref("T")).body(lambda b: b.compoundstatement(block([]))))),
    template([], lambda d: d.simpledeclaration(lambda s: s.specifiers([lambda x: x.classspecifier(
        lambda k: k.key("struct").name(lambda n: n.identifier(name("S"))).body(True))]))),
    lambda u: u.functiondefinition(lambda f: f.declarator(function_of("S")).initializers([
        lambda i: i.member(lambda n: n.identifier(name("x"))).initializer(lambda x: x.parenthesizedinitializer(
            lambda p: p.arguments([number(1)]))),
        lambda i: i.member(lambda n: n.identifier(name("Bases"))).initializer(lambda x: x.initializerlist(
            lambda l: l.items([]))).pack(True)]).body(lambda b: b.trystatement(lambda t: t.body(block([])).handlers([
                handler(ellipsis, [])])))),
    lambda u: u.functiondefinition(lambda f: f.declarator(function_of("T")).body(lambda b: b.defaultedbody(lambda x: x))),
    lambda u: u.functiondefinition(lambda f: f.specifiers([integer]).declarator(function_of("d")).body(
        lambda b: b.deletedbody(lambda x: x))),
    lambda u: u.functiondefinition(lambda f: f.specifiers([keyword("void")]).declarator(function_of("o"))
                                   .virt_specifiers(["override", "final"]).body(lambda b: b.compoundstatement(
                                       block([say("b")])))),
])
assert text == """\
inline namespace a::inline b {
    int x;
}
namespace {}
extern "C" {
    int y;
}
extern "C" int z;
export {
    int e;
}
export void g() {}
template <typename T>
void h() requires T {}
template <>
struct S {};
S() try : x(1), Bases{}... {} catch (...) {}
T() = default;
int d() = delete;
void o() override final {
    b();
}
"""
print("ok")

<a id="XPRT-04"></a>
## XPRT-04 · Classes: members one level in, access specifiers one level out, member declarators' `= 0`, virt-specifiers, `requires`, bit-fields and their initializers, a conditional's members in its `group`; a lambda's block and a class within a term, at the level of the statement that holds them

In [ ]:
member = lambda declarator: lambda m: m.memberdeclaration(lambda d: d.specifiers([integer]).declarators([  # noqa
    declarator]))
members = [
    lambda m: m.accessspecifier(lambda x: x.access("public")),
    member(lambda d: d.declarator(function_of("v")).pure(True).virt_specifiers(["override"])),
    member(lambda d: d.declarator(function_of("r")).requires(a)),
    member(lambda d: d.declarator(declared("bits")).bitfield(number(4)).initializer(equal(number(1)))),
    member(lambda d: d.declarator(declared("more")).bitfield(number(2)).initializer(lambda i: i.initializerlist(
        lambda l: l.items([number(3)])))),
    member(lambda d: d.bitfield(number(0))),
    lambda m: m.directive(lambda d: d.directive(lambda t: t.ifdefdirective(lambda x: x.name(name("A")).group(
        lambda g: g.cpp_membergroup(lambda y: y.members([member(lambda d: d.declarator(declared("a")))])))))),
    lambda m: m.accessspecifier(lambda x: x.access("private")),
    lambda m: m.functiondefinition(lambda f: f.specifiers([integer]).declarator(function_of("get")).body(
        lambda b: b.compoundstatement(block([lambda u: u.returnstatement(lambda s: s.value(ref("bits")))])))),
]
cls = lambda s: s.classspecifier(lambda k: k.key("class").name(lambda n: n.identifier(name("C"))).body(True).members(  # noqa
    lambda m: m.memberlist(lambda l: l.members(members))))
lambda_ = lambda t: t.lambdaexpression(lambda x: x.captures([lambda c: c.defaultcapture(lambda d: d.mode("&"))]).block(  # noqa
    lambda k: k.compoundstatement(block([lambda u: u.ifstatement(lambda s: s.condition(lambda c: c.expression(a))
                                                                  .consequence(compound([say("b")])))]))))
text = within("classes.cpp", [
    lambda u: u.simpledeclaration(lambda s: s.specifiers([cls])),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.callexpression(lambda x: x.function(lambda_)))),
])
assert text == """\
void f() {
    class C {
    public:
        int v() override = 0;
        int r() requires a;
        int bits : 4 = 1;
        int more : 2 {3};
        int : 0;
#ifdef A
        int a;
#endif
    private:
        int get() {
            return bits;
        }
    };
    [&] {
        if (a) {
            b();
        }
    }();
}
"""
print("ok")

<a id="XPRT-05"></a>
## XPRT-05 · The layout of 0.3's C++ source, as written there, but where the printer lays it out otherwise; each of what it lays out otherwise, printed as it is

In [ ]:
assert reprinted("export namespace geo::inline v1 {\n    struct Point {\n        double x = 0, y = 0;\n"
                 "        auto operator<=>(const Point &) const = default;\n    };\n}\n") == (
    "export namespace geo::inline v1 {\n    struct Point {\n        double x = 0, y = 0;\n"
    "        auto operator<=>(const Point&) const = default;\n    };\n}\n")  # an abstract declarator, unspaced
assert reprinted("template <typename T, int N = 3>\nrequires C<T>\nclass V {};\n") == (
    "template <typename T, int N = 3> requires C<T>\nclass V {};\n")  # the head's `requires` on its line
assert reprinted("T &operator[](int i) { return data[i]; }\n") == "T &operator[](int i) {\n    return data[i];\n}\n"
assert reprinted("void f() {\n    if (a) use(a);\n    else {\n    }\n}\n") == (
    "void f() {\n    if (a)\n        use(a);\n    else {}\n}\n")
assert reprinted("class D : public virtual B {};\n") == "class D : virtual public B {};\n"  # `virtual` first
assert reprinted("enum E {\n#ifdef A\n    B,\n#endif\n    C\n};\n") == "enum E {\n#ifdef A\n    B,\n#endif\n    C\n};\n"
assert reprinted("enum Color { Red, Green = 2, Blue, };\n") == "enum Color {\n    Red,\n    Green = 2,\n    Blue,\n};\n"
print("ok")

<a id="XPRT-06"></a>
## XPRT-06 · Invalid syntax prints as written, every kind prints alone, and what is no statement is refused

An `Error` where a statement or a member stands, its text as written; where an init-statement or a condition stands,
its text in its place. Every kind of `Statements.KINDS`, the clause `Handler` among them, and the `Error`, has a
printer (`Print.KINDS`); a kind none prints is refused, as is a file that is no translation unit, and a path the
workspace holds no file at.

In [ ]:
text = within("invalid.cpp", [
    lambda u: u.error(lambda e: e.text("if (a {\n  b;\n}").body([say("b")])),
    lambda u: u.ifstatement(lambda s: s.initializer(lambda i: i.error(lambda e: e.text("int = ;"))).condition(
        lambda c: c.error(lambda e: e.text("+"))).consequence(compound([]))),
])
assert text == "void f() {\n    if (a {\n  b;\n}\n    if (int = ; +) {}\n}\n"
build("alone.cpp", [variable("x", equal(number(1)))])
with store.transaction():
    (declaration,) = Objects.properties(store.singleton(Model.WORKSPACE).files({"path": "alone.cpp"}))["body"].values
    assert Print.print_statement(Terms.kind_of(declaration)[1]) == "int x = 1;"
    with raises(TypeError, match="no printer for the statement kind 'Programs.Cpp.InitDeclarator'"):
        Print.print_statement(Objects.properties(Terms.kind_of(declaration)[1])["declarators"].values[0])
    with raises(TypeError, match="not a Programs.Cpp.TranslationUnit"):
        Print.print_file(store.singleton(Model.WORKSPACE))
assert set(Print.KINDS) == {*Statements.KINDS, "Error"}
with raises(LookupError):
    Print.print_module(store, "nowhere.cpp")
assert printed("empty.cpp", []) == ""
print("ok")